<a href="https://colab.research.google.com/github/DiyaRana7/Flyrank_ML/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [9]:
!pip install -q datasets

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

## Unit of Analysis

For my lane (Refresh / Content Opportunity Scoring), one row represents the performance of one content page for one day.

Time Window

For this assignment I will use a single month of historical data (for example, March 2026) to analyze page performance before any future outcome.

This allows the model to learn only from information that would have been available at the decision time.

In [10]:
from datasets import load_dataset
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

daily = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    split="train[:10000]",
    token=HF_TOKEN
)

df = daily.to_pandas()

print("Rows:", len(df))
print("Columns:", len(df.columns))

df.head()

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Rows: 10000
Columns: 30


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,30,0,115.0,...,0,0,0,0,0,0,0,0,0,0
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,True,True,True,False,5,0,358.0,...,0,0,0,0,0,0,0,0,0,0
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,True,True,True,False,1,0,34.0,...,0,0,0,0,0,0,0,0,0,0
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,True,True,True,False,6,0,140.0,...,0,0,0,0,0,0,0,0,0,0
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,True,True,True,False,5,0,89.0,...,0,0,0,0,0,0,0,0,0,0


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## Fields

### Features

- impressions
- clicks
- ctr
- average_position
- engagement_rate

These are observable signals that are available before making the refresh decision.

### Label

The target is whether a page should be prioritized for refresh based on future performance decline.

### Context

- content_id
- client_hash_id
- date

These identify the page and provide context but are not prediction features.

### Excluded

I exclude future trend information and any label-derived fields because they would leak the answer into the model.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [11]:
print("Dataset Shape:", df.shape)

print("\nFirst Five Rows:")
display(df.head())

Dataset Shape: (10000, 30)

First Five Rows:


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,30,0,115.0,...,0,0,0,0,0,0,0,0,0,0
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,True,True,True,False,5,0,358.0,...,0,0,0,0,0,0,0,0,0,0
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,True,True,True,False,1,0,34.0,...,0,0,0,0,0,0,0,0,0,0
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,True,True,True,False,6,0,140.0,...,0,0,0,0,0,0,0,0,0,0
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,True,True,True,False,5,0,89.0,...,0,0,0,0,0,0,0,0,0,0


In [12]:
print("Unique Content Pages:", df["content_hash_id"].nunique())
print("Unique Clients:", df["client_hash_id"].nunique())

print("\nDate Range:")
print("Start:", df["report_date"].min())
print("End:", df["report_date"].max())

Unique Content Pages: 3521
Unique Clients: 3

Date Range:
Start: 2025-01-27
End: 2025-02-14


In [13]:
print("Top 10 Columns with Missing Values")

missing = df.isnull().sum().sort_values(ascending=False)

print(missing.head(10))

Top 10 Columns with Missing Values
gsc_avg_position      1
gsc_sum_position      1
content_hash_id       0
client_has_gsc        0
report_date           0
client_hash_id        0
gsc_data_available    0
client_has_ga4        0
gsc_impressions       0
ga4_data_available    0
dtype: int64


In [14]:
print("Rows with GSC Data Available:")
print(df[df["gsc_data_available"] == True].shape[0])

print("\nRows with GA4 Data Available:")
print(df[df["ga4_data_available"] == True].shape[0])

Rows with GSC Data Available:
10000

Rows with GA4 Data Available:
0


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Data Limits

This dataset contains historical search and analytics signals, but it cannot explain why Google's rankings change.

The data only shows observed performance metrics such as impressions, clicks, sessions, and engagement. It does not prove that one factor causes another.

Some rows may have missing Google Search Console (GSC) or Google Analytics 4 (GA4) data because not every client has access to both systems.

This project is intended to support human decisions about which pages to review. It should not be interpreted as a prediction of Google's ranking algorithm or as proof that refreshing a page will improve performance.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

## Self Check

- ✅ Every section is completed.
- ✅ Notebook runs without errors.
- ✅ No private client information is included.
- ✅ Claims are observational and decision-support only.
- ✅ Notebook will be committed to GitHub.